# IDRiD DR Classification — Architecture Comparison
### ResNet-50 and MobileNetV3-Large vs. EfficientNet-B3

**EfficientNet-B3 reference results (from main notebook):**
- Val QWK = 0.7664 | Test QWK = 0.6490 | 10.7M params

| Model | Params | ImageNet top-1 | Notes |
|---|---|---|---|
| EfficientNet-B3 *(reference)* | 10.7M | 81.1% | Main pipeline |
| ResNet-50 | 25.6M | 76.1% | Deeper residual; more capacity |
| MobileNetV3-Large | 5.5M | 75.3% | Lightweight; half the params of EffNet-B3 |


## 1. Imports & Setup

In [ ]:
import os, sys, random, logging
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, roc_auc_score, f1_score, classification_report, confusion_matrix
import torchvision.models as tv_models
import copy

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

NUM_CLASSES = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Device: {DEVICE}")

logging.basicConfig(
    level=logging.INFO,
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)


## 2. Preprocessing

In [ ]:
import cv2

def load_image_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None: raise FileNotFoundError(f"Cannot read: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def remove_black_borders(img, tol=7):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY); mask = gray > tol
    rows, cols = np.any(mask, axis=1), np.any(mask, axis=0)
    if not rows.any() or not cols.any(): return img
    y0, y1 = np.where(rows)[0][[0,-1]]; x0, x1 = np.where(cols)[0][[0,-1]]
    c = img[y0:y1+1, x0:x1+1]
    return c if c.shape[0] >= 10 and c.shape[1] >= 10 else img

def resize_image(img, target_size=224):
    return cv2.resize(img, (target_size, target_size),
                      interpolation=cv2.INTER_AREA if img.shape[0] > target_size
                      else cv2.INTER_LANCZOS4)

def fill_black_corners(img, tol=7, inpaint_radius=5):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    bm = (gray <= tol).astype(np.uint8) * 255
    return cv2.inpaint(img, bm, inpaint_radius, cv2.INPAINT_TELEA) if bm.sum() else img

def apply_clahe(img, clip_limit=2.0, tile_grid_size=(8, 8)):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB); l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

def ben_graham_preprocessing(img, sigma_fraction=1/30):
    sigma = max(img.shape[1] * sigma_fraction, 1.0)
    return cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0,0), sigmaX=sigma), -4, 128)

def preprocess_fundus_image(img, target_size=224):

    img = remove_black_borders(img)
    img = resize_image(img, target_size)
    img = fill_black_corners(img)
    img = apply_clahe(img)
    img = ben_graham_preprocessing(img)
    return img


## 3. Data Loading & Split

In [ ]:
def _find_dir_containing(root, *kws):
    from pathlib import Path
    if not Path(root).exists(): return None
    kws = [k.lower() for k in kws]
    cands = [p for p in Path(root).rglob("*")
             if p.is_dir() and all(k in p.name.lower() for k in kws)]
    return min(cands, key=lambda p: len(p.parts)) if cands else None

def _find_file_containing(root, *kws, suffix=".csv"):
    from pathlib import Path
    if not Path(root).exists(): return None
    kws = [k.lower() for k in kws]
    cands = [p for p in Path(root).rglob(f"*{suffix}")
             if all(k in p.name.lower() for k in kws)]
    return min(cands, key=lambda p: len(p.parts)) if cands else None

KAGGLE_INPUT = Path("/kaggle/input")
idrid_root = _find_dir_containing(KAGGLE_INPUT, "disease", "grading") or _find_dir_containing(KAGGLE_INPUT, "idrid") or _find_dir_containing(KAGGLE_INPUT, "diabetic", "retinopathy")
if idrid_root is None:
    raise FileNotFoundError("Could not auto-locate IDRiD dataset under /kaggle/input")
idrid_root = idrid_root.parent if "grading" in idrid_root.name.lower() else idrid_root
grading_root = _find_dir_containing(idrid_root, "disease", "grading") or idrid_root

train_images_dir = _find_dir_containing(grading_root, "train")
test_images_dir  = _find_dir_containing(grading_root, "test")
train_csv = (_find_file_containing(grading_root, "train", "label")
             or _find_file_containing(grading_root, "train"))
test_csv  = (_find_file_containing(grading_root, "test", "label")
             or _find_file_containing(grading_root, "test"))

print(f"Train images : {train_images_dir}")
print(f"Test  images : {test_images_dir}")
print(f"Train labels : {train_csv}")
print(f"Test  labels : {test_csv}")


In [ ]:
def _col(df, *kws):
    kws = [k.lower() for k in kws]
    for c in df.columns:
        if all(k in c.lower().replace("_"," ").strip() for k in kws): return c
    raise KeyError(f"No column matching {kws}")

def load_idrid_labels(csv_path):
    df = pd.read_csv(csv_path)
    df = df[[_col(df,"image"), _col(df,"retinopathy")]].copy()
    df.columns = ["image_name","dr_grade"]
    df["image_name"] = df["image_name"].astype(str).str.strip()
    df = df[df["image_name"] != ""].dropna().copy()
    df["dr_grade"] = df["dr_grade"].astype(int)
    return df.reset_index(drop=True)

train_df = load_idrid_labels(train_csv)
test_df  = load_idrid_labels(test_csv)

train_split_df, val_split_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df["dr_grade"], random_state=SEED
)
print(f"Train: {len(train_split_df)} | Val: {len(val_split_df)} | Test: {len(test_df)}")


## 4. Dataset & DataLoaders

In [ ]:
TARGET_SIZE = 224   # ResNet and MobileNet canonical input size

class IDRiDDataset(Dataset):
    def __init__(self, df, image_dir, target_size=TARGET_SIZE, ext=".jpg"):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.target_size = target_size
        self.ext = ext

    def __len__(self): return len(self.df)

    def _path(self, name):
        p = self.image_dir / f"{name}{self.ext}"
        if p.exists(): return p
        for e in (".jpg",".jpeg",".png",".tif",".tiff"):
            a = self.image_dir / f"{name}{e}"
            if a.exists(): return a
        return p

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = load_image_rgb(str(self._path(row["image_name"])))
        img = preprocess_fundus_image(img, self.target_size)
        img = torch.from_numpy(img.transpose(2,0,1).copy()).float() / 255.0
        return img, torch.tensor(int(row["dr_grade"]), dtype=torch.long)

BATCH_SIZE  = 16
NUM_WORKERS = 2

train_dataset = IDRiDDataset(train_split_df, train_images_dir)
val_dataset   = IDRiDDataset(val_split_df,   train_images_dir)
test_dataset  = IDRiDDataset(test_df,        test_images_dir)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

_imgs, _labels = next(iter(train_loader))
print(f"Batch shape: {tuple(_imgs.shape)} | labels: {_labels.tolist()}")


## 5. Loss Function

In [ ]:
def compute_class_weights(df, num_classes=NUM_CLASSES):
    counts = df["dr_grade"].value_counts().reindex(range(num_classes), fill_value=0).clip(lower=1)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights.values, dtype=torch.float32)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        loss = ((1 - torch.exp(-ce)) ** self.gamma) * ce
        if self.alpha is not None:
            a = self.alpha.to(logits.device)[targets]; loss = a * loss
            if self.reduction == "mean": return loss.sum() / a.sum()
        return loss.mean() if self.reduction == "mean" else loss

class_weights = compute_class_weights(train_split_df).to(DEVICE)
criterion = FocalLoss(alpha=class_weights, gamma=2.0)
print(f"FocalLoss | class weights: {[round(w,3) for w in class_weights.tolist()]}")


## 6. Models

In [ ]:
class ResNet50DR(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=True, dropout=0.3):
        super().__init__()
        weights = tv_models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        bb = tv_models.resnet50(weights=weights)

        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool)
        self.layer1 = bb.layer1
        self.layer2 = bb.layer2
        self.layer3 = bb.layer3
        self.last_block = bb.layer4
        self.avgpool = bb.avgpool
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(bb.fc.in_features, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.last_block(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.head(x)


class MobileNetV3DR(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=True, dropout=0.2):
        super().__init__()

        weights = (
            tv_models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
            if pretrained else None
        )
        bb = tv_models.mobilenet_v3_large(weights=weights)

        features = list(bb.features.children())
        self.stem = nn.Sequential(*features[:-1])
        self.last_block = nn.Sequential(features[-1])
        self.avgpool = bb.avgpool

        in_features = bb.classifier[0].in_features
        self.head = nn.Sequential(
            nn.Linear(in_features, 1280),
            nn.Hardswish(),
            nn.Dropout(dropout),
            nn.Linear(1280, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.last_block(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.head(x)

for name, m in [
    ("ResNet-50", ResNet50DR(pretrained=False)),
    ("MobileNetV3-Large", MobileNetV3DR(pretrained=False)),
]:
    m.to(DEVICE).eval()
    with torch.no_grad():
        out = m(_imgs.to(DEVICE))

    assert out.shape == (_imgs.shape[0], NUM_CLASSES), (
        f"Bad output shape: {out.shape}"
    )

    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name}: {n_params:,} params | output {tuple(out.shape)} ✓")

In [ ]:
from torchvision import models as tv_models

def build_efficientnet_b3(num_classes=NUM_CLASSES, pretrained=True, dropout=0.3):
    weights = tv_models.EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None
    m = tv_models.efficientnet_b3(weights=weights, dropout=dropout)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m

In [ ]:
import glob
matches = glob.glob("/kaggle/input/**/best_model.pt", recursive=True)
print("Found:", matches)

best_ckpt_path = Path(matches[0])

effnet = build_efficientnet_b3(pretrained=False).to(DEVICE)
best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
effnet.load_state_dict(best_ckpt["model_state_dict"])
print(f"EfficientNet-B3 loaded: epoch {best_ckpt['epoch']}, val_loss={best_ckpt['val_loss']:.4f}")

## 7. Training Loop

In [ ]:
PHASE1_EPOCHS       = 30
PHASE2_EPOCHS       = 20
PHASE1_LR           = 1e-3
PHASE2_LR_HEAD      = 1e-4
PHASE2_LR_BACKBONE  = 1e-5
WEIGHT_DECAY        = 1e-4
EARLY_STOP_PATIENCE = 5
UNFREEZE_FROM_STAGE = "last_block"   # attribute name on both models


def set_frozen(model, freeze_all=True):
    for p in model.parameters(): p.requires_grad = not freeze_all
    for p in model.head.parameters(): p.requires_grad = True


def set_phase2(model):
    for p in model.parameters(): p.requires_grad = False
    for p in model.last_block.parameters(): p.requires_grad = True
    for p in model.head.parameters(): p.requires_grad = True


class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience; self.best = float("inf")
        self.counter = 0; self.should_stop = False
    def step(self, v):
        if v < self.best - 1e-4: self.best = v; self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience: self.should_stop = True


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train(); total, n = 0.0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item() * imgs.size(0); n += imgs.size(0)
    return total / n


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval(); total, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        total += criterion(out, labels).item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item(); n += imgs.size(0)
    return total / n, correct / n


def run_phase(model, phase_name, num_epochs, optimizer, ckpt_path,
              epoch_offset=0, best_val=float("inf")):
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    stopper   = EarlyStopping(patience=EARLY_STOP_PATIENCE)
    last_ep   = epoch_offset

    print(f"=== {phase_name}: {num_epochs} epochs ===")
    for ep in range(1, num_epochs + 1):
        tl = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        vl, va = eval_epoch(model, val_loader, criterion, DEVICE)
        scheduler.step(vl); last_ep = epoch_offset + ep

        print(f"[{phase_name}][Ep {ep:3d}/{num_epochs}] "
                    f"train={tl:.4f} val={vl:.4f} acc={va:.4f}")

        if vl < best_val:
            best_val = vl
            torch.save({"epoch": last_ep, "model_state_dict": model.state_dict(),
                        "val_loss": vl}, ckpt_path)
            print(f"  -> best val_loss={vl:.4f}, saved")

        stopper.step(vl)
        if stopper.should_stop:
           print(f"  -> early stop at epoch {ep}"); break

    return best_val, last_ep


## 8. Evaluation Utilities

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval(); yt, yp, ypr = [], [], []
    for imgs, labels in loader:
        probs = F.softmax(model(imgs.to(device)), dim=1).cpu().numpy()
        yt.extend(labels.numpy()); yp.extend(probs.argmax(1)); ypr.extend(probs)
    return np.array(yt), np.array(yp), np.array(ypr)


def evaluate_full(model, loader, device, split_name="val"):
    y_true, y_pred, y_probs = get_predictions(model, loader, device)
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    f1  = f1_score(y_true, y_pred, average="macro")
    try:    auc = roc_auc_score(y_true, y_probs, multi_class="ovr", average="macro")
    except: auc = float("nan")
    cm     = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    report = classification_report(y_true, y_pred, zero_division=0,
                 target_names=[f"Grade {i}" for i in range(NUM_CLASSES)])
    print(f"=== {split_name} ===")
    print(f"QWK={qwk:.4f}  Macro F1={f1:.4f}  AUC-ROC={auc:.4f}")
    print(f"Per-class report:\n{report}")

    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels([f"G{i}" for i in range(NUM_CLASSES)])
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels([f"G{i}" for i in range(NUM_CLASSES)])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix — {split_name}")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i,j], ha="center", va="center",
                    color="white" if cm[i,j] > cm.max()/2 else "black")
    plt.colorbar(im); plt.tight_layout(); plt.show()
    return {"qwk": qwk, "f1": f1, "auc": auc, "cm": cm,
            "y_true": y_true, "y_pred": y_pred}

def count_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters()
               if (p.requires_grad or not trainable_only))

## 9. ResNet-50 — Train & Evaluate

Two-phase fine-tuning:
- **Phase 1:** freeze backbone, train head only (7,685 trainable params).
- **Phase 2:** unfreeze `layer4` + head (~6.6M trainable params).


In [ ]:
CKPT_DIR = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
ckpt_resnet = CKPT_DIR / "best_resnet50.pt"

resnet = ResNet50DR(pretrained=True).to(DEVICE)
best_val_resnet = float("inf")

set_frozen(resnet, freeze_all=True)
print(f"[ResNet-50 Phase 1] Trainable: {count_params(resnet, trainable_only=True):,}")
opt = optim.AdamW(filter(lambda p: p.requires_grad, resnet.parameters()),
                  lr=PHASE1_LR, weight_decay=WEIGHT_DECAY)
best_val_resnet, last_ep = run_phase(resnet, "ResNet-50 Phase 1",
                                      PHASE1_EPOCHS, opt, ckpt_resnet)

p1 = torch.load(ckpt_resnet, map_location=DEVICE, weights_only=False)
resnet.load_state_dict(p1["model_state_dict"])
print(f"Phase 1 best restored: epoch {p1['epoch']}, val_loss={p1['val_loss']:.4f}")

set_phase2(resnet)
print(f"[ResNet-50 Phase 2] Trainable: {count_params(resnet, trainable_only=True):,}")
opt = optim.AdamW([
    {"params": [p for p in resnet.last_block.parameters() if p.requires_grad],
     "lr": PHASE2_LR_BACKBONE},
    {"params": [p for p in resnet.head.parameters() if p.requires_grad],
     "lr": PHASE2_LR_HEAD},
], weight_decay=WEIGHT_DECAY)
best_val_resnet, _ = run_phase(resnet, "ResNet-50 Phase 2",
                                PHASE2_EPOCHS, opt, ckpt_resnet,
                                epoch_offset=last_ep, best_val=best_val_resnet)

print(f"ResNet-50 training complete. Best val_loss={best_val_resnet:.4f}")

In [ ]:
ckpt = torch.load(ckpt_resnet, map_location=DEVICE, weights_only=False)
resnet.load_state_dict(ckpt["model_state_dict"])
peint(f"Loaded checkpoint: epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")

resnet_val_metrics  = evaluate_full(resnet, val_loader,  DEVICE, "ResNet-50 validation")
resnet_test_metrics = evaluate_full(resnet, test_loader, DEVICE, "ResNet-50 TEST (final)")


## 10. MobileNetV3-Large — Train & Evaluate

In [ ]:
ckpt_mobile = CKPT_DIR / "best_mobilenetv3.pt"

mobilenet = MobileNetV3DR(pretrained=True).to(DEVICE)
best_val_mobile = float("inf")

# Phase 1
set_frozen(mobilenet, freeze_all=True)
print(f"[MobileNetV3 Phase 1] Trainable: {count_params(mobilenet, trainable_only=True):,}")
opt = optim.AdamW(filter(lambda p: p.requires_grad, mobilenet.parameters()),
                  lr=PHASE1_LR, weight_decay=WEIGHT_DECAY)
best_val_mobile, last_ep = run_phase(mobilenet, "MobileNetV3 Phase 1",
                                      PHASE1_EPOCHS, opt, ckpt_mobile)

p1 = torch.load(ckpt_mobile, map_location=DEVICE, weights_only=False)
mobilenet.load_state_dict(p1["model_state_dict"])
print(f"Phase 1 best restored: epoch {p1['epoch']}, val_loss={p1['val_loss']:.4f}")

set_phase2(mobilenet)
print(f"[MobileNetV3 Phase 2] Trainable: {count_params(mobilenet, trainable_only=True):,}")
opt = optim.AdamW([
    {"params": [p for p in mobilenet.last_block.parameters() if p.requires_grad],
     "lr": PHASE2_LR_BACKBONE},
    {"params": [p for p in mobilenet.head.parameters() if p.requires_grad],
     "lr": PHASE2_LR_HEAD},
], weight_decay=WEIGHT_DECAY)
best_val_mobile, _ = run_phase(mobilenet, "MobileNetV3 Phase 2",
                                PHASE2_EPOCHS, opt, ckpt_mobile,
                                epoch_offset=last_ep, best_val=best_val_mobile)

print(f"MobileNetV3 training complete. Best val_loss={best_val_mobile:.4f}")


In [ ]:
ckpt = torch.load(ckpt_mobile, map_location=DEVICE, weights_only=False)
mobilenet.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded checkpoint: epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")

mobile_val_metrics  = evaluate_full(mobilenet, val_loader,  DEVICE, "MobileNetV3 validation")
mobile_test_metrics = evaluate_full(mobilenet, test_loader, DEVICE, "MobileNetV3 TEST (final)")

## 11. Final Comparison

In [ ]:
EFFNET_RESULTS = {
    "val_qwk":  0.7664,   
    "test_qwk": 0.6490,   
    "val_f1":   0.4571,
    "test_f1":  0.4008,
    "val_auc":  0.8054,
    "test_auc": 0.7699,
    "params":   10_703_917,
    "input_size": 300,
}

results = {
    "EfficientNet-B3\n(main model)": {
        "Val QWK":   EFFNET_RESULTS["val_qwk"],
        "Test QWK":  EFFNET_RESULTS["test_qwk"],
        "Val F1":    EFFNET_RESULTS["val_f1"],
        "Test F1":   EFFNET_RESULTS["test_f1"],
        "Params":    EFFNET_RESULTS["params"],
        "Input":     EFFNET_RESULTS["input_size"],
    },
    "ResNet-50": {
        "Val QWK":  resnet_val_metrics["qwk"],
        "Test QWK": resnet_test_metrics["qwk"],
        "Val F1":   resnet_val_metrics["f1"],
        "Test F1":  resnet_test_metrics["f1"],
        "Params":   count_params(resnet),
        "Input":    TARGET_SIZE,
    },
    "MobileNetV3-Large": {
        "Val QWK":  mobile_val_metrics["qwk"],
        "Test QWK": mobile_test_metrics["qwk"],
        "Val F1":   mobile_val_metrics["f1"],
        "Test F1":  mobile_test_metrics["f1"],
        "Params":   count_params(mobilenet),
        "Input":    TARGET_SIZE,
    },
}

comp_df = pd.DataFrame(results).T
comp_df["Params"] = comp_df["Params"].apply(lambda x: f"{int(x):,}")
comp_df["Input"]  = comp_df["Input"].apply(lambda x: f"{int(x)}×{int(x)}")
print("ARCHITECTURE COMPARISON")
print(comp_df.to_string(float_format="%.4f"))
print()
best_model = comp_df["Test QWK"].astype(float).idxmax()
print(f"Best test QWK: {best_model} ({float(comp_df.loc[best_model,'Test QWK']):.4f})")

In [ ]:
models = list(results.keys())
val_qwks  = [results[m]["Val QWK"]  for m in models]
test_qwks = [results[m]["Test QWK"] for m in models]

x = np.arange(len(models)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w/2, val_qwks,  w, label="Val QWK",  color="#4c78a8")
ax.bar(x + w/2, test_qwks, w, label="Test QWK", color="#f58518")
ax.set_xticks(x); ax.set_xticklabels([m.replace("\n"," ") for m in models])
ax.set_ylabel("QWK (higher is better)")
ax.set_title("Validation vs Test QWK — Architecture Comparison")
ax.set_ylim(0, 1.0); ax.axhline(0.7, color="gray", linestyle="--", linewidth=0.8, label="QWK=0.7 reference")
ax.legend(); ax.grid(True, alpha=0.3, axis="y"); plt.tight_layout(); plt.show()


In [ ]:
resnet = ResNet50DR(pretrained=False).to(DEVICE)
ckpt = torch.load(ckpt_resnet, map_location=DEVICE, weights_only=False)
resnet.load_state_dict(ckpt["model_state_dict"])
print(f"ResNet-50 checkpoint: epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")
resnet_test_metrics = evaluate_full(resnet, test_loader, DEVICE, "ResNet-50 TEST (final)")

mobilenet = MobileNetV3DR(pretrained=False).to(DEVICE)
ckpt = torch.load(ckpt_mobile, map_location=DEVICE, weights_only=False)
mobilenet.load_state_dict(ckpt["model_state_dict"])
print(f"MobileNetV3 checkpoint: epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")
mobile_test_metrics = evaluate_full(mobilenet, test_loader, DEVICE, "MobileNetV3 TEST (final)")

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        if f.endswith(".pt"):
            print(os.path.join(root, f))

In [ ]:
effnet_val_metrics  = evaluate_full(effnet, val_loader,  DEVICE, "EfficientNet-B3 validation")
effnet_test_metrics = evaluate_full(effnet, test_loader, DEVICE, "EfficientNet-B3 TEST (final)")

resnet_val_metrics  = evaluate_full(resnet,    val_loader,  DEVICE, "ResNet-50 validation")
resnet_test_metrics = evaluate_full(resnet,    test_loader, DEVICE, "ResNet-50 TEST (final)")

mobile_val_metrics  = evaluate_full(mobilenet, val_loader,  DEVICE, "MobileNetV3 validation")
mobile_test_metrics = evaluate_full(mobilenet, test_loader, DEVICE, "MobileNetV3 TEST (final)")

In [ ]:
import json, shutil
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_to_save = {
    "EfficientNet-B3": {
        "val_qwk":  float(effnet_val_metrics["qwk"]),
        "test_qwk": float(effnet_test_metrics["qwk"]),
        "val_f1":   float(effnet_val_metrics["f1"]),
        "test_f1":  float(effnet_test_metrics["f1"]),
        "val_auc":  float(effnet_val_metrics["auc"]),
        "test_auc": float(effnet_test_metrics["auc"]),
    },
    "ResNet-50": {
        "val_qwk":  float(resnet_val_metrics["qwk"]),
        "test_qwk": float(resnet_test_metrics["qwk"]),
        "val_f1":   float(resnet_val_metrics["f1"]),
        "test_f1":  float(resnet_test_metrics["f1"]),
        "val_auc":  float(resnet_val_metrics["auc"]),
        "test_auc": float(resnet_test_metrics["auc"]),
    },
    "MobileNetV3-Large": {
        "val_qwk":  float(mobile_val_metrics["qwk"]),
        "test_qwk": float(mobile_test_metrics["qwk"]),
        "val_f1":   float(mobile_val_metrics["f1"]),
        "test_f1":  float(mobile_test_metrics["f1"]),
        "val_auc":  float(mobile_val_metrics["auc"]),
        "test_auc": float(mobile_test_metrics["auc"]),
    },
}

with open(OUTPUT_DIR / "comparison_metrics.json", "w") as f:
    json.dump(results_to_save, f, indent=2)
print(f"Metrics saved to {OUTPUT_DIR / 'comparison_metrics.json'}")

for model_name, val_m, test_m in [
    ("EfficientNet-B3", effnet_val_metrics,  effnet_test_metrics),
    ("ResNet-50",       resnet_val_metrics,  resnet_test_metrics),
    ("MobileNetV3",     mobile_val_metrics,  mobile_test_metrics),
]:
    for split_name, m in [("validation", val_m), ("test", test_m)]:
        cm = m.get("confusion_matrix", m.get("cm"))
        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(NUM_CLASSES))
        ax.set_xticklabels([f"Grade {i}" for i in range(NUM_CLASSES)])
        ax.set_yticks(range(NUM_CLASSES))
        ax.set_yticklabels([f"Grade {i}" for i in range(NUM_CLASSES)])
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(f"{model_name} — {split_name.capitalize()}\n"
                     f"QWK={m['qwk']:.4f}  F1={m['f1']:.4f}")
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                ax.text(j, i, cm[i, j], ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
        plt.colorbar(im); plt.tight_layout()
        fname = f"confusion_{model_name.lower().replace('-','').replace(' ','_')}_{split_name}.png"
        plt.savefig(OUTPUT_DIR / fname, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved {fname}")

print("\n" + "="*50)
print("FILES SAVED TO /kaggle/working/results/")
print("="*50)
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name:40s} {f.stat().st_size/1024:.1f} KB")